# PC Count Robustness — gpt2-xl L36 worddur
Shows that hippocampal semantic encoding results are stable across feature dimensionality.
Train/test R² curve demonstrates 100 PCs is well within the good-fit regime.

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

# ── CONFIG ──────────────────────────────────────────────────────────────────
MODEL_TAG  = 'gpt2-xl'
CTX_TAG    = '_ctx200'
WIN_TAG    = '_worddur'
PERM_TAG   = '_xshuffle'   # perm type used for sweep
LAYER      = 36
PC_COUNTS  = [5, 10, 20, 50, 100, 200, 300, 500]
REGIONS    = ['hippocampus']
CONDITIONS = ['self', 'other']
COLORS     = {'self': '#2166ac', 'other': '#d6604d'}
FIG_DIR    = '../figures'
GLM_BASE   = '/scratch/aniluchavez/ConvoDATAS/SemanticGLM'
os.makedirs(FIG_DIR, exist_ok=True)

def glm_dir(pc):
    return os.path.join(GLM_BASE,
                        f'{MODEL_TAG}{CTX_TAG}{WIN_TAG}{PERM_TAG}',
                        f'pc{pc}')

def load_pc(pc):
    d = glm_dir(pc)
    rows = []
    for f in sorted(glob.glob(os.path.join(d, f'*_L{LAYER:02d}_sem.pkl'))):
        obj = pickle.load(open(f, 'rb'))
        rows.append(obj['df'] if isinstance(obj, dict) else obj)
    if not rows:
        return None
    df = pd.concat(rows, ignore_index=True)
    df['n_components'] = pc
    return df

# load all available PC counts
dfs = {pc: load_pc(pc) for pc in PC_COUNTS}
available = [pc for pc, df in dfs.items() if df is not None]
print('Available PC counts:', available)
for pc in available:
    df = dfs[pc]
    has_train = 'r2_train' in df.columns
    has_sig   = df['significant'].any()
    print(f'  pc={pc:3d}: {len(df)} neurons  r2_train={has_train}  any_sig={has_sig}')

In [ ]:
# ── PLOT 1: Train vs Test R² curve (bias-variance tradeoff) ─────────────────
has_train = all('r2_train' in dfs[pc].columns for pc in available if dfs[pc] is not None)

fig, axes = plt.subplots(1, len(REGIONS), figsize=(6 * len(REGIONS), 5), squeeze=False)

for ax, region in zip(axes[0], REGIONS):
    for cond in CONDITIONS:
        c = COLORS[cond]
        pcs, med_train, med_test = [], [], []
        for pc in available:
            df = dfs[pc]
            sub = df[(df['region'] == region) & (df['condition'] == cond)]
            if sub.empty: continue
            pcs.append(pc)
            med_test.append(sub['r2'].median())
            if has_train:
                med_train.append(sub['r2_train'].median())

        ax.plot(pcs, med_test, color=c, marker='o', linewidth=2,
                markersize=7, label=f'{cond} test')
        if has_train:
            ax.plot(pcs, med_train, color=c, marker='s', linewidth=2,
                    linestyle='--', markersize=7, alpha=0.6, label=f'{cond} train')
            ax.fill_between(pcs, med_test, med_train, color=c, alpha=0.07)

    ax.axvline(100, color='gray', linewidth=1, linestyle=':', label='pc=100 (used)')
    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Number of PCs')
    ax.set_ylabel('Median McFadden R²')
    ax.set_title(f'{region}  — train vs test R²')
    ax.legend(fontsize=9, frameon=False)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_xticks(available)

plt.suptitle(f'gpt2-xl L{LAYER}  |  5-fold temporal block CV  |  train vs test R²', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/01_train_test_curve.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 2: % significant neurons vs PC count ────────────────────────────────
# Only shown for PC counts where perms were run (significant column has variance)
fig, axes = plt.subplots(1, len(REGIONS), figsize=(6 * len(REGIONS), 5), squeeze=False)

for ax, region in zip(axes[0], REGIONS):
    for cond in CONDITIONS:
        c = COLORS[cond]
        pcs, pct_sig = [], []
        for pc in available:
            df = dfs[pc]
            sub = df[(df['region'] == region) & (df['condition'] == cond)]
            if sub.empty: continue
            pcs.append(pc)
            pct_sig.append(100 * sub['significant'].mean())

        ax.plot(pcs, pct_sig, color=c, marker='o', linewidth=2,
                markersize=7, label=cond)

    ax.axhline(5, color='gray', linewidth=0.8, linestyle='--', label='5% chance')
    ax.axvline(100, color='gray', linewidth=1, linestyle=':', label='pc=100 (used)')
    ax.set_xlabel('Number of PCs')
    ax.set_ylabel('% significant neurons (FDR q<0.05)')
    ax.set_title(f'{region}  — significance vs PC count')
    ax.legend(fontsize=9, frameon=False)
    ax.set_ylim(0, None)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_xticks(available)

plt.suptitle(f'gpt2-xl L{LAYER}  |  xcirc null  |  robustness across PC count', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/01_pct_sig_vs_pc.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 3: Train/test ratio — how much are we overfitting? ─────────────────
if not has_train:
    print('r2_train not available')
else:
    fig, axes = plt.subplots(1, len(REGIONS), figsize=(6 * len(REGIONS), 5), squeeze=False)

    for ax, region in zip(axes[0], REGIONS):
        for cond in CONDITIONS:
            c = COLORS[cond]
            pcs, ratios = [], []
            for pc in available:
                df = dfs[pc]
                sub = df[(df['region'] == region) & (df['condition'] == cond)]
                if sub.empty: continue
                med_train = sub['r2_train'].median()
                med_test  = sub['r2'].median()
                ratio = med_test / med_train if med_train > 0 else np.nan
                pcs.append(pc)
                ratios.append(ratio)

            ax.plot(pcs, ratios, color=c, marker='o', linewidth=2,
                    markersize=7, label=cond)

        ax.axhline(1.0, color='k', linewidth=0.8, linestyle='--', label='train=test')
        ax.axvline(100, color='gray', linewidth=1, linestyle=':', label='pc=100 (used)')
        ax.set_xlabel('Number of PCs')
        ax.set_ylabel('test R² / train R²')
        ax.set_title(f'{region}  — test/train ratio')
        ax.legend(fontsize=9, frameon=False)
        ax.set_ylim(0, 1.2)
        ax.set_xscale('log')
        ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
        ax.set_xticks(available)

    plt.suptitle('test/train R² ratio — values near 1.0 indicate no overfitting', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/01_test_train_ratio.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
rows = []
for pc in available:
    df = dfs[pc]
    for region in REGIONS:
        for cond in CONDITIONS:
            sub = df[(df['region'] == region) & (df['condition'] == cond)]
            if sub.empty: continue
            med_train = sub['r2_train'].median() if 'r2_train' in sub.columns else np.nan
            med_test  = sub['r2'].median()
            rows.append({
                'n_components': pc,
                'region': region,
                'condition': cond,
                'n_neurons': len(sub),
                'pct_sig': round(100 * sub['significant'].mean(), 1),
                'med_r2_test': round(med_test, 4),
                'med_r2_train': round(med_train, 4),
                'test_train_ratio': round(med_test / med_train, 3) if med_train > 0 else np.nan,
            })

display(pd.DataFrame(rows))